# Stakeholder Gym — DPO on Colab single T4

Direct Preference Optimization on Qwen 2.5-3B. Bypasses the GRPO plateau (run-1 hit `clipped_ratio=1` at reward 0.12) by training on synthesized (prompt, chosen, rejected) pairs. **No rollouts during training.** ~30-40 min total on Colab T4.

Setup: **Runtime → Change runtime type → T4 GPU**.

Run All works. Don't skip cells.

In [ ]:
# Cell 1 — GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU not attached. Runtime -> Change runtime type -> T4 GPU.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  {p.total_memory/1e9:.1f}GB  compute {p.major}.{p.minor}')

In [ ]:
# Cell 2 — install Unsloth + TRL stack. Unsloth is single-GPU optimized,
# 2x faster than vanilla transformers on T4. Avoids accelerate setup.
!pip install -q unsloth trl datasets pydantic networkx pyyaml matplotlib

In [ ]:
# Cell 3 — fresh clone (delete any stale clone first; safe to rerun)
import os, subprocess
%cd /content
!rm -rf /content/meta
!git clone --depth 1 https://github.com/SAISriram19/meta.git /content/meta
%cd /content/meta
!git log --oneline -1
# Verify DPO files present
!ls scripts/build_dpo_pairs.py scripts/train_dpo_ddp.py 2>&1
!grep -n 'class DPOTrainer\|DPOConfig' scripts/train_dpo_ddp.py | head -3

In [ ]:
# Cell 4 — build DPO preference pairs from scenarios. ~5 sec.
!python scripts/build_dpo_pairs.py --out data/dpo_pairs.jsonl --cap-per-scenario 80
!wc -l data/dpo_pairs.jsonl

In [ ]:
# Cell 5 — BEFORE numbers (rule-based baselines, ~90s)
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled,memory_aware \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train_dpo

In [ ]:
# Cell 6 — load Qwen 2.5-3B with Unsloth 4-bit + LoRA
from unsloth import FastLanguageModel
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=64,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
tokenizer.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable/1e6:.2f}M / total: {total/1e6:.2f}M')

In [ ]:
# Cell 7 — load DPO pairs into dataset
import json
from datasets import Dataset
rows = []
with open('data/dpo_pairs.jsonl') as f:
    for line in f:
        d = json.loads(line)
        rows.append({'prompt': d['prompt'], 'chosen': d['chosen'], 'rejected': d['rejected']})
ds = Dataset.from_list(rows)
print(f'{len(ds)} pairs loaded')

In [ ]:
# Cell 8 — DPO TRAINING. Single T4. ~30-40 min.
# Defensive DPOConfig: drop kwargs the installed TRL version doesn't accept.
from trl import DPOConfig, DPOTrainer

base_kwargs = dict(
    output_dir='/content/outputs/dpo-stakeholder',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    beta=0.1,
    max_prompt_length=1500,
    max_length=1700,
    logging_steps=1,
    save_steps=50,
    seed=42,
    bf16=True,
    report_to='none',
    remove_unused_columns=False,
)
while True:
    try:
        config = DPOConfig(**base_kwargs)
        break
    except TypeError as e:
        import re
        m = re.search(r"unexpected keyword argument '(\w+)'", str(e))
        if not m:
            raise
        bad = m.group(1)
        print(f'[info] DPOConfig dropped: {bad}')
        base_kwargs.pop(bad, None)

# DPOTrainer kwargs: tokenizer -> processing_class around TRL 0.12. Try both.
tk = dict(model=model, train_dataset=ds, args=config)
try:
    trainer = DPOTrainer(processing_class=tokenizer, **tk)
except TypeError:
    trainer = DPOTrainer(tokenizer=tokenizer, **tk)

trainer.train()
model.save_pretrained('/content/outputs/dpo-stakeholder-lora')
tokenizer.save_pretrained('/content/outputs/dpo-stakeholder-lora')
print('training done')

In [ ]:
# Cell 9 — plot DPO curves (loss + margin + preference accuracy)
import matplotlib.pyplot as plt
log = trainer.state.log_history
steps = [h['step'] for h in log if 'loss' in h]
loss = [h['loss'] for h in log if 'loss' in h]
margins = [h.get('rewards/margins', None) for h in log if 'loss' in h]
acc = [h.get('rewards/accuracies', None) for h in log if 'loss' in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(steps, loss, marker='o', alpha=0.6, color='C0')
axes[0].set_title('DPO loss')
axes[0].set_xlabel('step')
axes[0].grid(alpha=0.3)
if any(m is not None for m in margins):
    axes[1].plot(steps, [m or 0 for m in margins], marker='o', alpha=0.6, color='C1')
    axes[1].set_title('reward margin (chosen - rejected)')
    axes[1].set_xlabel('step')
    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1].grid(alpha=0.3)
if any(a is not None for a in acc):
    axes[2].plot(steps, [a or 0 for a in acc], marker='o', alpha=0.6, color='C2')
    axes[2].set_title('preference accuracy')
    axes[2].set_xlabel('step')
    axes[2].set_ylim(0, 1)
    axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/outputs/dpo_training_curves.png', dpi=120)
plt.show()
print(f'loss first={loss[0]:.4f} last={loss[-1]:.4f}')
if any(m is not None for m in margins):
    last_m = next((m for m in reversed(margins) if m is not None), 0)
    print(f'final reward margin: {last_m:+.4f}')
if any(a is not None for a in acc):
    last_a = next((a for a in reversed(acc) if a is not None), 0)
    print(f'final preference accuracy: {last_a:.3f}')

In [ ]:
# Cell 10 — post-train eval. Load LoRA, run on L0+L2, 3 seeds. ~10 min.
import sys
from pathlib import Path
sys.path.insert(0, '.')
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT
from scripts.train import format_prompt, parse_completion

FastLanguageModel.for_inference(model)

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.4, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

cfg_eval = EvalConfig(
    policies={'dpo_qwen3b_colab': make_trained_policy()},
    scenarios=['L0_launch', 'L2_strategic_shift'],
    seeds=[0, 1, 2],
    out_dir=Path('eval_outputs/post_train_dpo'),
)
run_eval(cfg_eval, verbose=True)

In [ ]:
# Cell 11 — before/after comparison. Handles {'cells': [...]} unwrap.
import json
with open('eval_outputs/pre_train_dpo/summary.json') as f:
    pre = json.load(f)
with open('eval_outputs/post_train_dpo/summary.json') as f:
    post = json.load(f)
pre_rows = pre['cells'] if isinstance(pre, dict) and 'cells' in pre else pre
post_rows = post['cells'] if isinstance(post, dict) and 'cells' in post else post

print(f'{"kind":<7} {"policy":<22} {"scenario":<22} {"reward":>8} {"bad":>5} {"terminal":>8} {"princ":>6}')
print('-' * 92)
for r in pre_rows:
    print(f'BEFORE  {r["policy"]:<22} {r["scenario"]:<22} {r["total_reward_mean"]:>+8.3f} {r["bad_agreements_mean"]:>5.1f} {r["terminal_score_mean"]:>+8.3f} {r["principled_mean"]:>6.1f}')
print()
for r in post_rows:
    print(f'AFTER   {r["policy"]:<22} {r["scenario"]:<22} {r["total_reward_mean"]:>+8.3f} {r["bad_agreements_mean"]:>5.1f} {r["terminal_score_mean"]:>+8.3f} {r["principled_mean"]:>6.1f}')

In [ ]:
# Cell 12 — download artifacts to your laptop
from google.colab import files
import shutil, tarfile, os
out = '/content/outputs'
shutil.copy('eval_outputs/pre_train_dpo/summary.json', f'{out}/pre_train_dpo_summary.json')
shutil.copy('eval_outputs/post_train_dpo/summary.json', f'{out}/post_train_dpo_summary.json')
with tarfile.open(f'{out}/dpo-stakeholder-lora.tar.gz', 'w:gz') as t:
    t.add(f'{out}/dpo-stakeholder-lora', arcname='dpo-stakeholder-lora')
files.download(f'{out}/dpo_training_curves.png')
files.download(f'{out}/pre_train_dpo_summary.json')
files.download(f'{out}/post_train_dpo_summary.json')
files.download(f'{out}/dpo-stakeholder-lora.tar.gz')